In [ ]:
import pandas as pd
import numpy as np
import iqplot
import glob


import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')


In [ ]:
e003_metadata = pd.read_csv('e003_coalescence_metadata_round4_good.csv').set_index('sample')
e003_metadata_ins = e003_metadata.loc[e003_metadata['is_inoculumn'],:]
e003_metadata_ins =e003_metadata_ins.loc[e003_metadata_ins['parent_subjects'].isin(['AA-AA','AE-AE','AF-AF']),:]
good_samples = ['C4-e003Coalescence-mBHI-inoculumn-redo',
                'A2-e003Coalescence-Inoculumn-mGAM',
                'D5-e003Coalescence-Inoculumn-mGAM',
                'A2-e003Coalescence-mBHI-inoculumn-redo',
                'C4-e003Coalescence-Inoculumn-mGAM',
                'D5-e003Coalescence-Inoculumn-mBHI']
                
e003_metadata_ins =e003_metadata_ins.loc[good_samples,:]


In [ ]:

folders = glob.glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/calculateDiversityDepthv3/*')
all_dfs = []
for folder in folders:
    species_id = folder.split('/')[-1]
    fname1 = f'{folder}/{species_id}_med_depth_df.csv'
    df_depth = pd.read_csv(fname1).set_index('Unnamed: 0').rename(columns={'0': 'depth'})

    fname2 = f'{folder}/{species_id}_num_int_sites2.csv'
    df_num_int= pd.read_csv(fname2).set_index('Unnamed: 0').rename(columns={'0': 'num_int_sites'})
    try:

        fname3 = f'{folder}/{species_id}_diversity_df2.csv'
        df_div= pd.read_csv(fname3).set_index('Unnamed: 0').rename(columns={'0': 'diversity'})
    except:
        continue

    full_df = pd.concat([df_depth, df_num_int, df_div],axis=1)
    full_df['species_id'] = int(species_id)
   # full_df['species'] = df_metadata.loc[int(species_id), 'species'].split('s__')[-1]
    
    all_dfs.append(full_df)

    
all_dfs = pd.concat(all_dfs).reset_index().rename(columns = {'Unnamed: 0':'sample'})
all_dfs_coal = all_dfs.loc[all_dfs['sample'].isin(e003_metadata_ins.index.values),:]
all_dfs_coal = all_dfs_coal.loc[all_dfs_coal['depth']>5,:]
all_dfs_coal['subject'] = all_dfs_coal['sample'].transform(lambda x: e003_metadata_ins.loc[x,'parent_subjects'])
all_dfs_coal['media'] = all_dfs_coal['sample'].transform(lambda x: e003_metadata_ins.loc[x,'parent_media'])
all_dfs_coal['type_mesocosm'] = all_dfs_coal['sample'].transform(lambda x: e003_metadata_ins.loc[x,'type_mesocosm'])
all_dfs_coal['mesocosm'] = all_dfs_coal['sample'].transform(lambda x: e003_metadata_ins.loc[x,'mesocosm'])
all_dfs_coal = all_dfs_coal.loc[all_dfs_coal['depth']>=5,:]
all_dfs_coal 

In [ ]:
fnames = glob.glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/calculateFixedDiffsFastv3/*/*_fixed_diffs.csv')
all_dfs = []
for fname in fnames:
    df = pd.read_csv(fname).drop(columns='Unnamed: 0').rename(columns = {'index': 'sample2'})
    good_samples = df.loc[df['sample2'].isin(e003_metadata_ins.index.values),:]
    good_samples = good_samples.loc[good_samples['sample1'].isin(e003_metadata_ins.index.values),:]
    good_samples['species_id']=fname.split('/')[-2]
    species= fname.split('/')[-2]
 #   good_samples['species'] = df_metadata.loc[int(species), 'species'].split('s__')[-1]
    
  #  good_samples['parents1'] = good_samples['sample1'].transform(lambda x: e003_metadata.loc[e003_metadata['sample']==x,'parent_subjects'].values[0])
   # good_samples['parents2'] = good_samples['sample2'].transform(lambda x: e003_metadata.loc[e003_metadata['sample']==x,
    #                                                         'parent_subjects'].values[0])
    
    all_dfs.append(good_samples)
all_dfs = pd.concat(all_dfs)
good_samples = all_dfs
all_dfs.head()

In [ ]:
good_samples = all_dfs
good_samples['diversity']= good_samples['fixed_diffs']/good_samples['comparisons'] 
good_samples['parents1'] = good_samples['sample1'].transform(lambda x: e003_metadata_ins.loc[x,
                                                             'parent_subjects'])
good_samples['parents2'] = good_samples['sample2'].transform(lambda x: e003_metadata_ins.loc[x,
                                                             'parent_subjects'])

good_samples['passage1'] = good_samples['sample1'].transform(lambda x:e003_metadata_ins.loc[x,
                                                             'passage'])
good_samples['passage2'] = good_samples['sample2'].transform(lambda x: e003_metadata_ins.loc[x,
                                                             'passage'])

good_samples['media1'] = good_samples['sample1'].transform(lambda x: e003_metadata_ins.loc[x,
                                                             'parent_media'])
good_samples['media2'] = good_samples['sample2'].transform(lambda x: e003_metadata_ins.loc[x,
                                                             'parent_media'])

#good_samples=good_samples.loc[(good_samples['passage1']==5)*(good_samples['passage2']==5),:]
#good_samples=good_samples.loc[(good_samples['parents1']!='AC/PP')*(good_samples['parents2']!='AC/PP'),:]
#good_samples=good_samples.loc[good_samples['parents1']!=good_samples['parents2'],:]
good_samples['log_fixed_diffs']=np.log10(good_samples['fixed_diffs']+1)
#good_samples = good_samples.loc[good_samples['num_int_sites']<1e3,:]

In [ ]:
good_samples

In [ ]:
good_samples_diff_sub = good_samples.loc[(good_samples['parents1']!=good_samples['parents2']),:]
good_samples_diff_sub = good_samples_diff_sub.loc[(good_samples_diff_sub['media1']==good_samples_diff_sub['media2']),:]
all_dfs_coal['diversity_type']='Wtn'
good_samples_diff_sub['diversity_type']= 'Btw'


good_samples_diff_sub_s1 = good_samples_diff_sub.copy().rename(columns = {'sample1':'sample',
                                                                         'parents1': 'subject',
                                                                         'media1': 'media'})
good_samples_diff_sub_s2 = good_samples_diff_sub.copy().rename(columns = {'sample2':'sample',
                                                                         'parents2': 'subject',
                                                                         'media2': 'media'})
all_dfs_coal['diversity_type'] = 'Wtn'
good_samples_diff_sub_s2['counts']=1
good_samples_diff_sub_s2.groupby(['sample','species_id']).sum(numeric_only=True)

In [ ]:
all_div = pd.concat([all_dfs_coal[['species_id','diversity','diversity_type','sample','subject','media']],
                     good_samples_diff_sub_s1[['species_id','diversity','diversity_type','sample','subject','media']],
                  #   good_samples_diff_sub_s2[['species_id','diversity','diversity_type','sample','subject','media']]
                    ])

all_div=all_div.loc[~all_div['diversity'].isna(),:]
all_div['species-sample-compare']=all_div['species_id'].astype(str)+'-'+ all_div['subject']+'-'+all_div['media']+'-'+all_div['diversity_type']
all_div['species-sample']=all_div['species_id'].astype(str)+'-'+ all_div['subject']+'-'+all_div['media']

all_div['species-sample']=all_div['species_id'].astype(str)+'-'+ all_div['sample']
all_divWt = all_div.loc[all_div['diversity_type']=='Wtn','species-sample'].unique()
all_divBt = all_div.loc[all_div['diversity_type']=='Btw','species-sample'].unique()
good_compares = np.intersect1d(all_divWt,all_divBt)
good_compares

In [ ]:

all_div_compare = all_div.loc[all_div['species-sample'].isin(good_compares),:]

all_div_compare_wtn = all_div_compare.loc[all_div_compare['diversity_type']=='Wtn',:]
all_div_compare_btw = all_div_compare.loc[all_div_compare['diversity_type']=='Btw',:]

p = hv.Points(all_div_compare_wtn.sort_values(by='diversity',ascending=False),
              kdims = ['species-sample','diversity'],vdims = ['diversity','species-sample']).opts(xrotation=60,width=800,
                                                                                           height= 400,size=3,
                                                           logy=True,
                                                           #logy=True,#ylim=(1,10**5),
                                                                        jitter = None,
                                                                                      # xaxis=None,
                                                                        cmap = bokeh.palettes.Light[8],
                                                               xlabel='Species',xticks=None,
                                                                        show_legend=True,
                                                                        legend_position='right',  
                                                                                     #  multi_level=False,
                                                               ylabel='Diversity',
                                                              color='#F39019')

p2 = hv.Points(all_div_compare_btw,
              kdims = ['species-sample','diversity'],vdims = ['diversity','species-sample']).opts(xrotation=60,width=600,
                                                                                           height= 400,size=3,
                                                           logy=True,
                                                           #logy=True,#ylim=(1,10**5),
                                                                        jitter = None,
                                                                                       xaxis=None,
                                                                        cmap = bokeh.palettes.Light[8],
                                                               xlabel='Species',xticks=None,
                                                                        show_legend=True,
                                                                        legend_position='right',  
                                                                                     #  multi_level=False,
                                                               ylabel='Diversity',
                                                              color='#00882B')

p=(p*p2).opts(height=250,width=800)
p= hv.render(p)
p.output_backend = "svg"
bokeh.io.show(p)
test_name='bloh'
bokeh.io.export_svgs (p, filename = test_name + '.svg')

In [ ]:
all_div_compare_wtn.sort_values(by='diversity',ascending=False)
bads = all_div_compare_wtn.loc[all_div_compare_wtn['diversity']>=1e-3,'species-sample-compare'].unique()
bads_bads = ['101346-AE-AF-mBHI','101346-AA-AF-mBHI', '101346-AA-AE-mBHI', '102506-AE-AF-mBHI']
np.sort(bads)

In [ ]:
len(all_div_compare_wtn['species-sample-compare']) - 15

In [ ]:
len(all_div_compare_wtn['species_id'].unique())

In [ ]:

p.output_backend = "svg"
bokeh.io.show(p)
test_name='bloh'
bokeh.io.export_svgs (p, filename = test_name + '.svg')
import svglib.svglib as svglib
from reportlab.graphics import renderPDF
!rsvg-convert -f pdf -o file2.pdf bloh.svg